In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import tensorflow_datasets as tfds
import tensorflow as tf

import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv('/content/drive/MyDrive/DPL/Project/Data/balanced_spam.csv')

In [4]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)

In [5]:
print("Train set:",train_df.shape)
print("Test set:",test_df.shape)

Train set: (12626, 2)
Test set: (5412, 2)


In [6]:
train_dataset = tf.data.Dataset.from_tensor_slices((train_df['message'].values, train_df['label'].values))
test_dataset = tf.data.Dataset.from_tensor_slices((test_df['message'].values, test_df['label'].values))
train_dataset.element_spec

(TensorSpec(shape=(), dtype=tf.string, name=None),
 TensorSpec(shape=(), dtype=tf.int64, name=None))

In [7]:
BUFFER_SIZE = 10000
BATCH_SIZE = 64
train_dataset = train_dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [8]:
for example, label in train_dataset.take(1):
  print('texts: ', example.numpy()[:5])
  print()
  print('labels: ', label.numpy()[:5])

texts:  [b'PRIVATE! Your 2004 Account Statement for 07742676969 shows 786 unredeemed Bonus Points. To claim call 08719180248 Identifier Code: 45239 Expires'
 b"see, i knew giving you a break a few times woul lead to you always wanting to miss curfew. I was gonna gibe you 'til one, but a MIDNIGHT movie is not gonna get out til after 2. You need to come home. You need to getsleep and, if anything, you need to b studdying ear training."
 b'Free Top ringtone -sub to weekly ringtone-get 1st week free-send SUBPOLY to 81618-?3 per week-stop sms-08718727870'
 b'Boltblue tones for 150p Reply POLY# or MONO# eg POLY3 1. Cha Cha Slide 2. Yeah 3. Slow Jamz 6. Toxic 8. Come With Me or STOP 4 more tones txt MORE'
 b'Welcome to UK-mobile-date this msg is FREE giving you free calling to 08719839835. Future mgs billed at 150p daily. To cancel send \\go stop\\" to 89123"']

labels:  [1 0 1 1 1]


In [9]:
VOCAB_SIZE = 1000
encoder = tf.keras.layers.TextVectorization(
    max_tokens=VOCAB_SIZE)
encoder.adapt(train_dataset.map(lambda text, label: text))

In [10]:
vocab = np.array(encoder.get_vocabulary())
vocab[:20]

array(['', '[UNK]', 'to', 'you', 'a', 'i', 'call', 'the', 'your', 'u',
       'for', 'is', 'and', 'free', '2', 'now', 'or', 'on', 'have', 'in'],
      dtype='<U36')

In [11]:
i = 0
for ec in encoder(example)[:3].numpy():
  print(example[i].numpy())
  print(ec)
  i += 1
  print()

b'PRIVATE! Your 2004 Account Statement for 07742676969 shows 786 unredeemed Bonus Points. To claim call 08719180248 Identifier Code: 45239 Expires'
[209   8 525 162 225  10   1 124 940 242 222 227   2  38   6   1 245 131
   1 228   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0]

b"see, i knew giving you a break a few times woul lead to you always wanting to miss curfew. I was gonna gibe you 'til one, but a MIDNIGHT movie is not gonna get out til after 2. You need to come home. You need to getsleep and, if anything, you need to b studdying ear training."
[105   5   1   1   3   4 665   4 651 981   1   1   2   3 605   1   2 261
   1   5  99 419   1   3 926 118  56   4   1 647  11  48 419  29  45 926
 236  14   3 107   2 114 148   3 107   2   1  12  53 382   3 107   2 182
   1   1   1]

b'Free Top ringtone -sub to weekly ringtone-get 1st week free-send SUBPOLY to 81618-?3 per week-stop sms

In [81]:
model = tf.keras.Sequential([
    encoder,
    tf.keras.layers.Embedding(
        input_dim=len(encoder.get_vocabulary()),
        output_dim=16,
        mask_zero=True),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

In [82]:
# Compile the Keras model to configure the training process:
model.compile(loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
              optimizer=tf.keras.optimizers.Adam(1e-4),
              metrics=['accuracy'])

## Train the model

In [83]:
history = model.fit(train_dataset, epochs=10)

Epoch 1/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 10s 29ms/step - accuracy: 0.7041 - loss: 0.6812
Epoch 2/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 6s 28ms/step - accuracy: 0.9483 - loss: 0.2733
Epoch 3/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 6s 29ms/step - accuracy: 0.9709 - loss: 0.1137
Epoch 4/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 6s 29ms/step - accuracy: 0.9762 - loss: 0.0775
Epoch 5/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 6s 29ms/step - accuracy: 0.9801 - loss: 0.0642
Epoch 6/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 6s 29ms/step - accuracy: 0.9872 - loss: 0.0515
Epoch 7/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 6s 28ms/step - accuracy: 0.9877 - loss: 0.0400
Epoch 8/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 6s 29ms/step - accuracy: 0.9900 - loss: 0.0296
Epoch 9/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 6s 29ms/step - accuracy: 0.9902 - loss: 0.0297
Epoch 10/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 6s 29ms/step - accuracy: 0.9923 - loss: 0.0239


In [84]:
test_loss, test_acc = model.evaluate(test_dataset)

print('Test Loss:', test_loss)
print('Test Accuracy:', test_acc)

85/85 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9887 - loss: 0.0352
Test Loss: 0.032756295055150986
Test Accuracy: 0.989098310470581


In [55]:
# Định nghĩa mẫu văn bản dưới dạng một chuỗi đơn
sample_text = ('Then we wait 4 u lor... No need 2 feel bad lar...')

# Chuyển đổi mẫu văn bản thành định dạng mà lớp encoder yêu cầu
sample_text_tensor = tf.constant([sample_text])

# Dự đoán
predictions = model.predict(sample_text_tensor)
print(predictions)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 284ms/step
[[1.5672756e-05]]
